<a href="https://colab.research.google.com/github/jaehyeon0420/agent_tutorial/blob/master/rag_basic_example/rag_retriever_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 이 파일은 text-embedding-3-large, bge-m3, BM25, Ensemble 모델의 RAG 성능을 평가합니다.
### 2026년 2월에 발표된 대한민국 인공지능 행동계획.pdf를 사용합니다.

In [ ]:
# OpenAI API, Vector DB, PDF 처리, 토큰 계산
!pip install -U openai chromadb pypdf tiktoken sentence-transformers

In [ ]:
import os
import json
from typing import List, Dict

import torch
from sentence_transformers import SentenceTransformer

# OpenAI API 관련
from openai import OpenAI

# 벡터 DB 관련
import chromadb
from chromadb.utils import embedding_functions

# PDF 로딩 관련
from pypdf import PdfReader


#### PDF의 모든 텍스트를 하나의 문자열로 병합합니다.

In [ ]:
def load_pdf(file_path: str) -> str:
    reader = PdfReader(file_path)
    full_text = ""

    for page in reader.pages:
        # 페이지 텍스트 추출 (None 반환 대비)
        text = page.extract_text()
        if text:
            full_text += text + "\n"

    return full_text

pdf_text = load_pdf("대한민국 인공지능 행동계획.pdf")

In [ ]:
pdf_text[:300]

'대한민국 인공지능 행동계획\n2026. 2. 25\n국가인공지능전략위원회\n- 2 -\n≪들어가며≫인공지능(AI)은 과학적 발견의 방식을 근본적으로 바꾸고, 산업의 구조를 다시 짜며, 인간의 사고와 소통, 학습의 영역까지 재정의하고 있습니다. 이 거대한 기술혁명은 세계 질서를 재편하고, 경제와 안보의 규칙을 새로 만듭니다. 우리는 반도체·이동 통신·자동차 산업에서 그랬던 것처럼, 인공지능(AI)을 새로운 도약의 발판으로 삼아 온 국민이 혜택을 얻는 인공지능 기본사회를 이루고자 합니다. 능동적이고 자주적인 대응으로 이 변화를 우리의 것으로 '

#### 전체 문자열을 오버랩 기준으로 청킹하여 하나의 문자열 리스트를 생성합니다

In [ ]:
def chunk_text(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    chunks = []

    # 시작 지점을 overlap만큼 이동하며 반복
    start = 0

    while start < len(text):
        # 종료 지점 계산
        end = start + chunk_size

        # 청킹
        chunk = text[start:end]
        chunks.append(chunk)

        # 다음 시작 지점 = 현재 종료 지점 - 오버랩
        start += (chunk_size - chunk_overlap)

        # 무한 루프 방지 (오버랩이 청크 사이즈보다 크거나 같을 경우)
        if chunk_size <= chunk_overlap:
            break

    return chunks

# 설정 값 예시
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

chunks = chunk_text(pdf_text, CHUNK_SIZE, CHUNK_OVERLAP)

In [ ]:
chunks[0]

'대한민국 인공지능 행동계획\n2026. 2. 25\n국가인공지능전략위원회\n- 2 -\n≪들어가며≫인공지능(AI)은 과학적 발견의 방식을 근본적으로 바꾸고, 산업의 구조를 다시 짜며, 인간의 사고와 소통, 학습의 영역까지 재정의하고 있습니다. 이 거대한 기술혁명은 세계 질서를 재편하고, 경제와 안보의 규칙을 새로 만듭니다. 우리는 반도체·이동 통신·자동차 산업에서 그랬던 것처럼, 인공지능(AI)을 새로운 도약의 발판으로 삼아 온 국민이 혜택을 얻는 인공지능 기본사회를 이루고자 합니다. 능동적이고 자주적인 대응으로 이 변화를 우리의 것으로 만드는 것은 우리의 과제입니다. 「대한민국 인공지능 행동계획」은 이러한 비전을 실현하기 위해 마련되었습니다. 이 계획은 단순한 기술정책이 아니라, 인공지능을 새로운 국가 성장엔진으로 삼아 산업을 혁신하고, 국민의 삶을 개선하며, 인류 공동의 번영에 기여하고자 합니다. 행동계획은 세가지 정책축과 12대 전략분야로 구성되었습니다:먼저 첫 번째 축은 인공지능 혁신 생태계 조성입니다.인공지능 경쟁의 승패는 일차적으로 인프라 역량의 보유 규모와 활용 역량에 좌우됩니다. 대한민국은 데이터, 컴퓨팅, 반도체, 전력 등 인공지능 개발･활용의 기초가 되는 국가 인프라인 AI고속도로를 구축하고, 이를 바탕으로 인공지능 생태계 전반의 혁신에 나서야 합니다. 또한 초거대 인공지능 모델과 핵심기술을 선점하고, 세계 최고 수준의 인공지능 인재를 양성하여 글로벌 인공지능 기술 개발의 맨 앞에 설 수 있어야 합니다. 이를 위해 역량 있는 인공지능 연구자와 기업들이 규제에 가로막히지 않고 혁신에 도전할 수 있도록 법·제도·행정 전반의 구조를 재편해야 합니다. 이러한 측면에서 첫 번째 축은 다섯 가지 전략분야로 구성됩니다: ① AI고속도로 구축, ② 차세대 인공지능 기술 선점, ③ 인공지능 핵심인재 확보, ④ 독자 범용 인공지능 모델 확보, ⑤ 인공지능 규제혁신입니다. 이 5가지 분야는 인공지능 강국의 토대를 마련하고, 다음 세대의 국가 경쟁력을 결정짓는 

In [ ]:
chunks[1]

' 핵심인재 확보, ④ 독자 범용 인공지능 모델 확보, ⑤ 인공지능 규제혁신입니다. 이 5가지 분야는 인공지능 강국의 토대를 마련하고, 다음 세대의 국가 경쟁력을 결정짓는 출발점이 될 것입니다.\n- 3 -\n두 번째 축은 범국가 인공지능 기반 대전환입니다.인공지능은 더 이상 특정 산업의 도구가 아니라, 국가 전체의 시스템을 바꾸는 원천입니다. 대한민국은 인공지능과 함께 산업, 공공, 지역, 국방, 문화 등 모든 국가영역의 혁신을 이룩해야 합니다. 인공지능은 제조·에너지·금융·바이오·의료 등 주력 산업의 효율성을 극대화하고, 행정과 복지, 교육 등 공공서비스의 품질을 획기적으로 개선할 수 있습니다. 또한 지역거점별 인공지능 허브 구축을 통해 수도권과 지방의 격차를 줄이고, 인공지능을 활용한 지능형 방위·안보 체계를 통해 첨단 국방 강국으로 도약해야 합니다. 아울러 우리가 가진 세계적인 장점인 K-문화와 K-콘텐츠산업에도 인공지능을 결합하여 창의성과 기술이 융합된 K-문화 르네상스를 열어가야 합니다. 이러한 측면에서 두 번째 축은 다섯 가지 전략분야로 구성됩니다: ⑥ 산업 인공지능 대전환, ⑦ 공공 인공지능 대전환, ⑧ 지역 인공지능 대전환, ⑨ 인공지능기반 문화강국, ⑩ 인공지능기반 국방강국 도약입니다. 이 5가지 전략 분야는 인공지능을 통해 국가 체계를 전면적으로 혁신하는 대한민국 인공지능 대전환 프로젝트입니다.마지막 축은 글로벌 인공지능 기본사회 기여입니다.인공지능 시대의 경쟁은 기술의 경쟁을 넘어 가치와 신뢰의 경쟁으로 진화하고 있습니다. 대한민국은 민주주의, 인권, 포용의 가치를 바탕으로 인공지능 기본사회를 실현하고, 국제사회가 신뢰할 수 있는 인공지능 거버넌스 모델을 제시해야 합니다. 인공지능이 국민 누구에게나 보편적 혜택으로 작동하고, 사회적 약자와 지역, 세대 간 격차를 줄이는 포용적 인공지능 복지 시스템을 구축하는 한편, 글로벌 인공지능 협력의 허브로서, 인공지능기술·데이터·인재 협력을 선도하고, 국제규범 논의와 공동 연구개발을 통해 세계 인공지

In [ ]:
chunks[-1]

' 데이터·모델·서비스·에이전트가 국가, 산업, 플랫폼 간 자유롭게 상호작용할 수 있도록 하는 제도적·기술적·거버넌스 기반을 구축한다는 의미. 여기에는 기술적/데이터/신뢰 및 거버넌스/경제 및 정책적 상호운용성 등이 모두 포괄\n'

In [ ]:
import uuid
from google.colab import userdata

# 1. 클라이언트 및 로컬 모델 초기화
openai_client = OpenAI(api_key=userdata.get('OPEN_AI_KEY'))

# BGE-M3 로드 (GPU 사용 가능한 경우 cuda 설정)
device = "cuda" if torch.cuda.is_available() else "cpu"
bge_model = SentenceTransformer('BAAI/bge-m3', device=device)

# 2. ChromaDB 설정
db_client = chromadb.PersistentClient(path="./DB")

# 두 모델을 위한 별도의 컬렉션 생성
openai_collection = db_client.get_or_create_collection(name="collection_openai_large")
bge_collection = db_client.get_or_create_collection(name="collection_bge_m3")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

#### 청크 리스트를 text-embedding-3-large를 이용하여 임베딩 후 Chroma에 저장합니다

In [ ]:
def add_chunks_to_multi_db(chunks: list[str], metadata_list: list[dict] = None):
    """
    동일한 청크를 OpenAI와 BGE-M3 모델로 각각 임베딩하여 별도 컬렉션에 적재합니다.
    """

    # 동일한 청크는 두 DB에서 같은 ID를 가지게 하여 추후 비교 평가를 용이하게 합니다.
    ids = [str(uuid.uuid4()) for _ in range(len(chunks))]
    if metadata_list is None:
        metadata_list = [{"source": "pdf_document"} for _ in range(len(chunks))]

    # text-embedding-3-large로 임베딩하여 적재
    print(f"[*] OpenAI 임베딩 생성 중")
    oa_response = openai_client.embeddings.create(
        input=chunks,
        model="text-embedding-3-large"
    )
    oa_embeddings = [item.embedding for item in oa_response.data]

    openai_collection.add(
        ids=ids,
        embeddings=oa_embeddings,
        documents=chunks,
        metadatas=metadata_list
    )
    print(f"    -> OpenAI 컬렉션 적재 완료.")

    # --- C. BAAI/bge-m3 적재 ---
    print(f"[*] BGE-M3 임베딩 생성 중...")

    # 로컬 모델을 사용한 배치 임베딩 생성
    bge_embeddings = bge_model.encode(
        chunks,
        normalize_embeddings=True,
        show_progress_bar=True
    ).tolist()

    bge_collection.add(
        ids=ids,
        embeddings=bge_embeddings,
        documents=chunks,
        metadatas=metadata_list
    )
    print(f"    -> BGE-M3 컬렉션 적재 완료.")

    print(f"\n[성공] 총 {len(chunks)}개의 청크가 두 벡터 DB에 각각 저장되었습니다.")

In [ ]:
add_chunks_to_multi_db(chunks)

[*] OpenAI 임베딩 생성 중
    -> OpenAI 컬렉션 적재 완료.
[*] BGE-M3 임베딩 생성 중...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

    -> BGE-M3 컬렉션 적재 완료.

[성공] 총 205개의 청크가 두 벡터 DB에 각각 저장되었습니다.


## RAG 평가
#### RAG 평가를 위해 gpt-4o를 사용하여 각 chunk당 3개의 질문 데이터를 생성합니다.
#### 각 chunk와 생성된 질문은 연결되어 있으며, chunk는 질문데 대한 '정답'으로 간주됩니다.

In [ ]:
import os
import re
import time
from tqdm import tqdm
from openai import OpenAI

def generate_evaluation_dataset(chunks, num_questions_per_doc=3):
    """
    각 청크(Chunk)에 대한 질문을 생성하고 평가용 데이터셋 구조를 반환합니다.

    Args:
        chunks: 청크 리스트
        num_questions_per_doc: 청크 당 생성할 질문 수

    Returns:
        all_queries: {질문_ID: 질문_텍스트}
        corpus: {문서_ID: 문서_텍스트}
        relevant_docs: {질문_ID: [관련_문서_ID]}
    """
    client = OpenAI(api_key=userdata.get('OPEN_AI_KEY'))

    all_queries = {}
    corpus = {}
    relevant_docs = {}

    prompt_template = """
    당신은 질문 생성 전문가입니다. 아래 제공된 [맥락]을 읽고, 해당 내용에 대해 답할 수 있는 질문을 {num}개 생성하세요.

    [맥락]
    {context}

    [지침]
    - 질문은 반드시 한국어로 작성하세요.
    - 각 질문은 독립적이어야 하며, 본문 없이 질문만 읽어도 의미가 전달되어야 합니다.
    - '위 본문에 따르면', '제시된 글에서'와 같은 표현은 절대 사용하지 마세요.
    - 질문만 작성하고 답안이나 해설은 포함하지 마세요.
    - 아래 형식과 같이 번호를 나열하여 생성하세요:
    1. 질문 내용
    2. 질문 내용
    """

    for i, chunk in enumerate(tqdm(chunks, desc="질문 생성 중")):
        time.sleep(1.0)

        # 입력 형식이 문자열인지 딕셔너리인지 확인하여 처리
        if isinstance(chunk, dict):
            text = chunk.get('text', '')
            doc_id = chunk.get('id', f"doc_{i}")
        else:
            text = chunk
            doc_id = f"doc_{i}"

        corpus[doc_id] = text

        try:
            # LLM 호출 (온도는 0으로 설정하여 일관성 유지)
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": "당신은 질문 생성 도우미입니다. 제공된 정보 기반으로 질문을 생성하세요."},
                    {"role": "user", "content": prompt_template.format(context=text, num=num_questions_per_doc)}
                ],
                temperature=0.7
            )

            # 응답 텍스트 파싱
            raw_output = response.choices[0].message.content.strip()
            lines = raw_output.split("\n")

            q_count = 0
            for line in lines:
                # 숫자. 형식을 제거하고 실제 질문 텍스트만 추출
                clean_question = re.sub(r"^\d+[\.\)\s]+", "", line).strip()

                if clean_question and len(clean_question) > 5:
                    query_id = f"q_{doc_id}_{q_count}"
                    all_queries[query_id] = clean_question
                    relevant_docs[query_id] = [doc_id]
                    q_count += 1

                    if q_count >= num_questions_per_doc:
                        break

        except Exception as e:
            print(f"\n[오류] 문서 {doc_id} 처리 중 오류 발생: {e}")

    return all_queries, corpus, relevant_docs

In [ ]:
# queries : 'q_문서번호_질문번호'를 key로, '생성된 질문'을 value로 가지는 딕셔너리
# corpus : 'doc_문서번호'를 key로, 문서를 value로 가지는 딕셔너리
# relevant_docs : 'q_문서번호_질문번호'를 key로, 'doc_문서번호'를 value로 가지는 딕셔너리
queries, corpus, relevant_docs = generate_evaluation_dataset(chunks)
print(f"생성된 질문 수: {len(queries)}")

질문 생성 중: 100%|██████████| 205/205 [07:53<00:00,  2.31s/it]

생성된 질문 수: 615


In [ ]:
# 생성된 질문 몇 개 샘플 확인
print("\n생성된 질문 샘플:")
for i, (query_id, query_text) in enumerate(list(queries.items())[:6]):
    print(f"질문 {i+1}: {query_text}")
    related_doc_ids = list(relevant_docs[query_id])
    print(f"  관련 문서 ID: {related_doc_ids[0]}")
    print(f"  문서 내용 일부: {corpus[related_doc_ids[0]]}...\n")


생성된 질문 샘플:
질문 1: 대한민국 인공지능 행동계획에서 제시된 첫 번째 정책축의 주요 목표는 무엇인가요?
  관련 문서 ID: doc_0
  문서 내용 일부: 대한민국 인공지능 행동계획
2026. 2. 25
국가인공지능전략위원회
- 2 -
≪들어가며≫인공지능(AI)은 과학적 발견의 방식을 근본적으로 바꾸고, 산업의 구조를 다시 짜며, 인간의 사고와 소통, 학습의 영역까지 재정의하고 있습니다. 이 거대한 기술혁명은 세계 질서를 재편하고, 경제와 안보의 규칙을 새로 만듭니다. 우리는 반도체·이동 통신·자동차 산업에서 그랬던 것처럼, 인공지능(AI)을 새로운 도약의 발판으로 삼아 온 국민이 혜택을 얻는 인공지능 기본사회를 이루고자 합니다. 능동적이고 자주적인 대응으로 이 변화를 우리의 것으로 만드는 것은 우리의 과제입니다. 「대한민국 인공지능 행동계획」은 이러한 비전을 실현하기 위해 마련되었습니다. 이 계획은 단순한 기술정책이 아니라, 인공지능을 새로운 국가 성장엔진으로 삼아 산업을 혁신하고, 국민의 삶을 개선하며, 인류 공동의 번영에 기여하고자 합니다. 행동계획은 세가지 정책축과 12대 전략분야로 구성되었습니다:먼저 첫 번째 축은 인공지능 혁신 생태계 조성입니다.인공지능 경쟁의 승패는 일차적으로 인프라 역량의 보유 규모와 활용 역량에 좌우됩니다. 대한민국은 데이터, 컴퓨팅, 반도체, 전력 등 인공지능 개발･활용의 기초가 되는 국가 인프라인 AI고속도로를 구축하고, 이를 바탕으로 인공지능 생태계 전반의 혁신에 나서야 합니다. 또한 초거대 인공지능 모델과 핵심기술을 선점하고, 세계 최고 수준의 인공지능 인재를 양성하여 글로벌 인공지능 기술 개발의 맨 앞에 설 수 있어야 합니다. 이를 위해 역량 있는 인공지능 연구자와 기업들이 규제에 가로막히지 않고 혁신에 도전할 수 있도록 법·제도·행정 전반의 구조를 재편해야 합니다. 이러한 측면에서 첫 번째 축은 다섯 가지 전략분야로 구성됩니다: ① AI고속도로 구축, ② 차세대 인공지능 기술 선점, ③ 인공지능 핵심인재 확

### 임베딩 리트리버, BM25 리트리버, 앙상블 리트리버를 생성합니다.


In [ ]:
!pip install rank_bm25

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from typing import List, Dict, Any, Optional

# --- [BM25 리트리버] ---
class BM25Retriever:
    """
    단어 빈도 기반의 BM25 알고리즘을 사용하여 문서를 검색하는 클래스입니다.
    """
    # chunks를 입력받아 토큰화하여 초기화 합니다.
    def __init__(self, documents: List[Dict[str, Any]], k: int = 5):
        self.documents = documents
        self.k = k
        self.tokenized_corpus = [doc['page_content'].split(" ") for doc in documents]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

    # 키워드 기반 상위 k개 문서를 조회합니다.
    def invoke(self, query: str, k: Optional[int] = None) -> List[Dict[str, Any]]:
        target_k = k if k else self.k
        tokenized_query = query.split(" ")
        top_n = self.bm25.get_top_n(tokenized_query, self.documents, n=target_k)
        return top_n


In [ ]:
# --- [Dense 리트리버 - ChromaDB용 Wrapper] ---
class DenseRetriever:
    """
    이미 구축된 ChromaDB 컬렉션을 사용하여 벡터 검색을 수행하는 클래스입니다.
    """
    def __init__(self, collection, client, model_name="text-embedding-3-large", k: int = 5):
        self.collection = collection
        self.client = client
        self.model_name = model_name
        self.k = k

    def invoke(self, query: str, k: Optional[int] = None) -> List[Dict[str, Any]]:
        target_k = k if k else self.k
        # 1. 쿼리 임베딩
        query_vector = self.client.embeddings.create(
            input=[query.replace("\n", " ")],
            model=self.model_name
        ).data[0].embedding

        # 2. ChromaDB 검색
        results = self.collection.query(
            query_embeddings=[query_vector],
            n_results=target_k
        )

        # 3. 데이터 포맷 통일
        docs = []
        for i in range(len(results['documents'][0])):
            docs.append({
                "page_content": results['documents'][0][i],
                "metadata": results['metadatas'][0][i] or {}
            })
        return docs

In [ ]:
# --- [BGEM3 리트리버 - ChromaDB용 Wrapper] ---
class BGEM3Retriever:
    """
    BAAI/bge-m3 모델을 사용하여 로컬에서 쿼리를 임베딩하고
    ChromaDB 검색을 수행하는 클래스입니다.
    """
    def __init__(self, collection, model_name="BAAI/bge-m3", k: int = 5):
        self.collection = collection
        self.k = k

        # GPU 사용 가능 여부 확인 (코랩 GPU 환경 권장)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # 1. 모델 로드
        print(f"[{model_name}] 모델을 {self.device}에 로드 중...")
        self.model = SentenceTransformer(model_name, device=self.device)
        print("모델 로드 완료.")

    def invoke(self, query: str, k: Optional[int] = None) -> List[Dict[str, Any]]:
        target_k = k if k else self.k

        # 2. 쿼리 임베딩 생성 (로컬 연산)
        # BGE-M3는 기본적으로 dense 임베딩을 제공합니다.
        query_vector = self.model.encode(
            query,
            normalize_embeddings=True, # 코사인 유사도 계산을 위해 정규화
            convert_to_numpy=True
        ).tolist()

        # 3. ChromaDB 검색
        results = self.collection.query(
            query_embeddings=[query_vector],
            n_results=target_k
        )

        # 4. 데이터 포맷 통일
        docs = []
        for i in range(len(results['documents'][0])):
            docs.append({
                "page_content": results['documents'][0][i],
                "metadata": results['metadatas'][0][i] or {}
            })
        return docs

# --- 실행 예시 ---
# bge_retriever = BGEM3Retriever(collection=collection, k=5)
# results = bge_retriever.invoke("삼성전자 전략에 대해 알려줘")

In [ ]:
import hashlib

# --- [Ensemble 리트리버] ---
class EnsembleRetriever:
    """
    BM25와 Dense 검색 결과를 RRF(Reciprocal Rank Fusion) 알고리즘으로 통합합니다.
    """
    def __init__(self, retrievers: List[Any], weights: List[float] = None, c: int = 60):
        # 각 리트리버의 가중치 설정 (기본값은 동일 가중치)
        self.retrievers = retrievers
        self.weights = weights if weights else [1.0] * len(retrievers)
        self.c = c # RRF 파라미터 (낮은 순위의 급격한 점수 하락 방지)

    def _get_doc_id(self, doc: Dict[str, Any]) -> str:
        """
        문서 내용의 공백을 제거하고 해싱하여 절대적인 고유 ID를 생성합니다.
        메타데이터 ID가 불확실할 때 가장 안전한 방법입니다.
        """
        # 텍스트의 앞뒤 공백을 제거하여 내용이 같으면 같은 ID가 나오게 함
        clean_content = doc['page_content'].strip()
        return hashlib.md5(clean_content.encode()).hexdigest()

    def invoke(self, query: str, k: int = 5) -> List[Dict[str, Any]]:
        all_results = [retriever.invoke(query, k=k*2) for retriever in self.retrievers]

        # RRF 점수 계산용 딕셔너리
        rrf_scores: Dict[str, Dict[str, Any]] = {}

        for i, docs in enumerate(all_results):
            weight = self.weights[i]
            for rank, doc in enumerate(docs):
                doc_id = self._get_doc_id(doc)
                # RRF 공식: 점수 = 가중치 / (순위 + C)
                score = weight / (rank + 1 + self.c)

                if doc_id not in rrf_scores:
                    rrf_scores[doc_id] = {"doc": doc, "score": score}
                else:
                    rrf_scores[doc_id]["score"] += score

        # 점수 기준 내림차순 정렬
        sorted_results = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)
        return [item["doc"] for item in sorted_results[:k]]

In [ ]:
chunks[0]

'대한민국 인공지능 행동계획\n2026. 2. 25\n국가인공지능전략위원회\n- 2 -\n≪들어가며≫인공지능(AI)은 과학적 발견의 방식을 근본적으로 바꾸고, 산업의 구조를 다시 짜며, 인간의 사고와 소통, 학습의 영역까지 재정의하고 있습니다. 이 거대한 기술혁명은 세계 질서를 재편하고, 경제와 안보의 규칙을 새로 만듭니다. 우리는 반도체·이동 통신·자동차 산업에서 그랬던 것처럼, 인공지능(AI)을 새로운 도약의 발판으로 삼아 온 국민이 혜택을 얻는 인공지능 기본사회를 이루고자 합니다. 능동적이고 자주적인 대응으로 이 변화를 우리의 것으로 만드는 것은 우리의 과제입니다. 「대한민국 인공지능 행동계획」은 이러한 비전을 실현하기 위해 마련되었습니다. 이 계획은 단순한 기술정책이 아니라, 인공지능을 새로운 국가 성장엔진으로 삼아 산업을 혁신하고, 국민의 삶을 개선하며, 인류 공동의 번영에 기여하고자 합니다. 행동계획은 세가지 정책축과 12대 전략분야로 구성되었습니다:먼저 첫 번째 축은 인공지능 혁신 생태계 조성입니다.인공지능 경쟁의 승패는 일차적으로 인프라 역량의 보유 규모와 활용 역량에 좌우됩니다. 대한민국은 데이터, 컴퓨팅, 반도체, 전력 등 인공지능 개발･활용의 기초가 되는 국가 인프라인 AI고속도로를 구축하고, 이를 바탕으로 인공지능 생태계 전반의 혁신에 나서야 합니다. 또한 초거대 인공지능 모델과 핵심기술을 선점하고, 세계 최고 수준의 인공지능 인재를 양성하여 글로벌 인공지능 기술 개발의 맨 앞에 설 수 있어야 합니다. 이를 위해 역량 있는 인공지능 연구자와 기업들이 규제에 가로막히지 않고 혁신에 도전할 수 있도록 법·제도·행정 전반의 구조를 재편해야 합니다. 이러한 측면에서 첫 번째 축은 다섯 가지 전략분야로 구성됩니다: ① AI고속도로 구축, ② 차세대 인공지능 기술 선점, ③ 인공지능 핵심인재 확보, ④ 독자 범용 인공지능 모델 확보, ⑤ 인공지능 규제혁신입니다. 이 5가지 분야는 인공지능 강국의 토대를 마련하고, 다음 세대의 국가 경쟁력을 결정짓는 

In [ ]:
# 1. 문자열 리스트인 chunks를 딕셔너리 리스트로 변환
documents = [
   {"page_content": chunk, "metadata": {"id": f"doc_{i}"}}
    for i, chunk in enumerate(chunks)
]

# 2. BM25용 문서 리스트 준비 (변환된 데이터 사용)
bm25 = BM25Retriever(documents=documents, k=5)

# 3. Dense용 리트리버 준비
dense = DenseRetriever(collection=openai_collection, client=openai_client, k=5)

# 4. bge-m3 리트리버 준비
bgem3 = BGEM3Retriever(collection=bge_collection, k=5)

# 5. 앙상블-1 설정 (bm25 + text-embedding-3-large)
ensemble1 = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])

# 6. 앙상블-2 설정 (bm25 + bge-m3)
ensemble2 = EnsembleRetriever(retrievers=[bm25, bgem3], weights=[0.5, 0.5])

print("리트리버 앙상블 구축 완료")

[BAAI/bge-m3] 모델을 cpu에 로드 중...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

모델 로드 완료.
리트리버 앙상블 구축 완료


### 검색 평가 메트릭 함수(Accuracy@k, Precision@k, Recall@k, MRR@k, NDCG@k, MAP@k)를 정의합니다.

In [ ]:
# 문서 ID와 내용 매핑 만들기 (평가 함수 외부에서 한 번만 실행)
document_mapping = {}
for i, doc in enumerate(documents):
    doc_id = doc['metadata']['id']
    document_mapping[doc['page_content']] = doc_id

# 내용으로 문서 ID 찾는 함수
def get_doc_id_by_content(content):
    """내용으로 문서 ID 찾기 (완전 일치 또는 부분 일치)"""
    # 완전 일치 확인
    if content in document_mapping:
        return document_mapping[content]

    # 부분 일치 확인 (내용이 더 길거나 짧을 수 있음)
    for doc_content, doc_id in document_mapping.items():
        if content in doc_content or doc_content in content:
            return doc_id

    return None

#### 1. Accuracy@k
Accuracy@k는 각 질문에 대한 검색 결과 K개에서 정답의 포함여부를 평가하는 지표입니다.
결과 K개에서 1개라도 정답이 포함되었다면 '성공'으로 간주됩니다.

예시) 질문에 대해 상위 5개의 검색 결과가 나왔다고 가정합니다.
- 질문1 : [**정답**, 오답, 오답, 오답, 오답] => 정답 포함되어 성공
- 질문2 : [오답, 오답, **정답**, 오답, 오답] => 정답 포함되어 성공
- 질문3 : [오답, 오답, 오답, 오답, 오답] => 정답 미포함이므로 실패

성공인 경우 1로, 실패인 경우 0으로 계산합니다.

Accuracy@5 = (1+1+0) / 3 = 0.666... => 66.6%

Accuracy@5 = 66.6% 해석 : 전체 질문 중 약 66.6%에서 상위 5개의 결과 안에 정답이 하나라도 포함되어 있음.

In [ ]:
def calculate_accuracy_at_k(retrieved_docs, relevant_docs, k):
    """
    상위 k개 결과 안에 정답이 포함되어 있는지 여부를 계산합니다.

    Args:
        retrieved_docs: 검색된 문서 목록
        relevant_docs: 관련 문서 ID 집합
        k: 상위 몇 개 결과를 고려할지

    Returns:
        0 또는 1 (정답이 상위 k개 안에 없으면 0, 있으면 1)
    """
    # 상위 k개만 고려
    top_k_docs = retrieved_docs[:k] if k <= len(retrieved_docs) else retrieved_docs

    # 상위 k개 중 정답이 하나라도 있는지 확인
    for doc in top_k_docs:
        # 메타데이터에서 ID 추출 시도
        doc_id = doc['metadata'].get('id')

        # 메타데이터에 ID가 없으면 내용으로 찾기
        if not doc_id:
            doc_id = get_doc_id_by_content(doc['page_content'])

        # 정답에 포함되어 있는지 확인
        if doc_id in relevant_docs:
            return 1.0

    return 0.0

#### 2. Precision@k
Precision@k는 각 질문에 대한 검색 결과 K개에서 정답 갯수를 평가하는 지표입니다.
결과 K개 중, 정답이 차지하는 비율을 측정합니다.

예시) 질문에 대해 상위 5개의 검색 결과가 나왔다고 가정합니다.
- 질문1 : [**정답**, 오답, 오답, 오답, 오답] => (1/5) = 0.2
- 질문2 : [오답, 오답, **정답**, 오답, 오답] => (1/5) = 0.2
- 질문3 : [오답, 오답, 오답, 오답, 오답] => (0/5) = 0

Precision@5 = (0.2+0.2+0) / 3 = 0.1333... => 13.3%

Precision@5 = 13.3% 해석 : 전체 질문에 대하여, 정답의 비율

In [ ]:
def calculate_precision_at_k(retrieved_docs, relevant_docs, k):
    """
    상위 k개 결과 중 정답의 비율을 계산합니다.

    Args:
        retrieved_docs: 검색된 문서 목록
        relevant_docs: 관련 문서 ID 집합
        k: 상위 몇 개 결과를 고려할지

    Returns:
        0~1 사이의 값 (상위 k개 중 정답의 비율)
    """
    # 상위 k개만 고려
    top_k_docs = retrieved_docs[:k] if k <= len(retrieved_docs) else retrieved_docs

    if not top_k_docs:
        return 0.0

    # 상위 k개 중 정답의 개수
    relevant_count = 0
    for doc in top_k_docs:
        # 메타데이터에서 ID 추출 시도
        doc_id = doc['metadata'].get('id')

        # 메타데이터에 ID가 없으면 내용으로 찾기
        if not doc_id:
            doc_id = get_doc_id_by_content(doc['page_content'])

        # 정답에 포함되어 있는지 확인
        if doc_id in relevant_docs:
            relevant_count += 1

    return relevant_count / len(top_k_docs)

#### 3. Recall@k
Recall@k는 검색 결과가 얼마나 포괄적으로 정답을 포함하고 있는지를 평가합니다.
전체 정답 중, 검색 결과 상위 k개 안에 포함된 정답의 비율을 나타냅니다.

상위 K개에 포함된 정답의 갯수를 전체 정답 갯수로 나눕니다.
각 질문당 실제 정답이 하나씩만 있는 경우 Accuracy와 Recall 값은 동일합니다.

예시) 질문에 대해 실제 정답 갯수가 2개라고 가정합니다.
- 질문1 : [**정답**, 오답, **정답**, 오답, 오답] => (2/2) = 1.0
- 질문2 : [오답, 오답, **정답**, 오답, 오답] => (1/2) = 0.5
- 질문3 : [오답, 오답, 오답, 오답, 오답] => (0/2) = 0

Recall@5 = (1.0+0.5+0) / 3 = 0.5... => 50.0%

Recall@5 = 50% 해석 : 전체 정답 중 50%가 상위 5개 결과에 포함되었음.
- 높을수록 검색 시스템이 관련 문서를 빠짐없이 찾아낸다
- 중요 정보를 놓치지 않아야 하는 상황에서 중요한 지표
- 모든 관련 정보를 찾는 능력을 평가하는 지표

In [ ]:
def calculate_recall_at_k(retrieved_docs, relevant_docs, k):
    """
    전체 정답 중 상위 k개에 포함된 정답의 비율을 계산합니다.

    Args:
        retrieved_docs: 검색된 문서 목록
        relevant_docs: 관련 문서 ID 집합
        k: 상위 몇 개 결과를 고려할지

    Returns:
        0~1 사이의 값 (전체 정답 중 상위 k개에 포함된 비율)
    """
    if not relevant_docs:
        return 0.0

    # 상위 k개만 고려
    top_k_docs = retrieved_docs[:k] if k <= len(retrieved_docs) else retrieved_docs

    # 상위 k개에 포함된 정답의 개수 계산
    relevant_found = 0
    for doc in top_k_docs:
        # 메타데이터에서 ID 추출 시도
        doc_id = doc['metadata'].get('id')

        # 메타데이터에 ID가 없으면 내용으로 찾기
        if not doc_id:
            doc_id = get_doc_id_by_content(doc['page_content'])

        # 정답에 포함되어 있는지 확인
        if doc_id in relevant_docs:
            relevant_found += 1

    return relevant_found / len(relevant_docs)

### 4. MRR@k (Mean Reciprocal Rank - 첫 번째 정답 순위의 역수)

MRR@k는 첫 번째 정답이 등장한 순위의 역수를 계산합니다.
첫 정답이 상위에 있을수록 높은 점수를 부여합니다.

계산 방법:
- 정답이 1위에 있으면 MRR = 1.0
- 정답이 2위에 있으면 MRR = 0.5
- 정답이 3위에 있으면 MRR = 0.33...
- 상위 k개 안에 정답이 없으면 MRR = 0

해석 예시:
- MRR@10 = 0.5: 첫 번째 정답이 평균적으로 2위에 등장한다는 의미
- 사용자가 정답을 얼마나 "빨리" 찾을 수 있는지를 평가하는 지표

실제 예시:
질문에 대해 상위 10개의 검색 결과가 다음과 같다고 가정합니다.
- 질문 1: [**정답**, 오답, 오답, ...] Reciprocal Rank = 1/1 = 1.0
- 질문 2: [오답, **정답**, 오답, ...] Reciprocal Rank = 1/2 = 0.5
- 질문 3: [오답, 오답, **정답**, ...] Reciprocal Rank = 1/3 = 0.33
- 질문 4: [오답, 오답, 오답, ...] Reciprocal Rank = 0 (정답이 상위 10위 안에 없음)

평균 MRR@10 = (1.0 + 0.5 + 0.33 + 0)/4 = 0.458

In [ ]:
def calculate_mrr_at_k(retrieved_docs, relevant_docs, k):
    """
    Mean Reciprocal Rank를 계산합니다 (첫 번째 정답 순위의 역수).

    Args:
        retrieved_docs: 검색된 문서 목록
        relevant_docs: 관련 문서 ID 집합
        k: 상위 몇 개 결과를 고려할지

    Returns:
        0~1 사이의 값 (첫 번째 정답 순위의 역수, 없으면 0)
    """
    # 상위 k개만 고려
    top_k_docs = retrieved_docs[:k] if k <= len(retrieved_docs) else retrieved_docs

    # 첫 번째 정답 순위 찾기
    for i, doc in enumerate(top_k_docs):
        # 메타데이터에서 ID 추출 시도
        doc_id = doc['metadata'].get('id')

        # 메타데이터에 ID가 없으면 내용으로 찾기
        if not doc_id:
            doc_id = get_doc_id_by_content(doc['page_content'])

        # 정답에 포함되어 있는지 확인
        if doc_id in relevant_docs:
            return 1.0 / (i + 1)  # 0부터 시작하므로 +1

    return 0.0

### 5. NDCG@k (Normalized Discounted Cumulative Gain)

NDCG@k는 검색 결과의 관련성과 순위를 모두 고려하는 메트릭입니다.  
관련 문서가 높은 순위에 배치될수록 높은 점수를 부여합니다.

계산 방법:
1. DCG(Discounted Cumulative Gain) 계산:
   - 검색된 각 문서가 정답인지 확인합니다.
   - 정답인 문서에 대해, 위치에 따라 할인된 점수를 부여합니다(log₂(rank+1)로 나눔).
   - 이 점수들의 합을 구합니다.

2. Ideal DCG 계산:
   - 모든 정답 문서가 상위에 있는 이상적인 경우의 DCG를 계산합니다.

3. NDCG = DCG / Ideal DCG (0~1 사이의 값)

해석 예시:
- NDCG@10 = 0.85: 검색 결과가 이상적인 순서에 85% 근접함
- 정답의 순위 분포를 고려하는 종합적인 메트릭

실제 예시:
- 질문 1: [정답, 정답, 오답] → NDCG = 1.0 (정답이 모두 상위에 있음)
- 질문 2: [오답, 정답, 오답] → NDCG ≈ 0.63 (정답이 두 번째 위치에 있음)
- 질문 3: [오답, 오답, 정답] → NDCG ≈ 0.39 (정답이 세 번째 위치에 있음)

평균 NDCG@3 = (1.0 + 0.63 + 0.39)/3 ≈ 0.673

In [ ]:
def calculate_ndcg_at_k(retrieved_docs, relevant_docs, k):
    """
    NDCG@k (Normalized Discounted Cumulative Gain)를 계산합니다.

    Args:
        retrieved_docs: 검색된 문서 목록
        relevant_docs: 관련 문서 ID 집합
        k: 상위 몇 개 결과를 고려할지

    Returns:
        0~1 사이의 값 (1에 가까울수록 이상적인 검색 결과)
    """
    # 상위 k개만 고려
    top_k_docs = retrieved_docs[:k] if k <= len(retrieved_docs) else retrieved_docs

    # 실제 DCG 계산
    dcg = 0
    for i, doc in enumerate(top_k_docs):
        # 메타데이터에서 ID 추출 시도
        doc_id = doc['metadata'].get('id')

        # 메타데이터에 ID가 없으면 내용으로 찾기
        if not doc_id:
            doc_id = get_doc_id_by_content(doc['page_content'])

        # 정답에 포함되어 있는지 확인
        if doc_id in relevant_docs:
            # i는 0부터 시작하므로 i+1이 실제 순위
            dcg += 1.0 / np.log2(i + 2)  # log_2(rank + 1)

    # 이상적인 DCG 계산 (모든 관련 문서가 상위에 있을 경우)
    ideal_dcg = 0
    for i in range(min(len(relevant_docs), k)):
        ideal_dcg += 1.0 / np.log2(i + 2)

    # NDCG 계산
    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0

### 6. MAP@k (Mean Average Precision)

MAP@k는 각 정답을 찾을 때마다의 Precision 값을 계산하여 평균을 낸 값입니다.
검색 결과의 전반적인 정확도와 일관성을 평가합니다.

계산 방법:
1. 각 정답 발견 시 해당 위치까지의 Precision 계산
2. 이 Precision 값들의 평균 산출

해석 예시:
- MAP@100 = 0.818: 상위 100개 내에서 정답을 찾을 때마다 계산된 Precision의 평균이 0.818
- 검색 시스템의 전반적인 일관성을 측정하는 종합적인 지표

실제 예시:
상위 5개 결과가 [**정답**, 오답, **정답**, 오답, **정답**]인 경우:
- 첫 번째 정답 발견 시 Precision = 1/1 = 1.0
- 두 번째 정답 발견 시 Precision = 2/3 = 0.67
- 세 번째 정답 발견 시 Precision = 3/5 = 0.6
- MAP@5 = (1.0 + 0.67 + 0.6) / 3 = 0.76

In [ ]:
def calculate_map_at_k(retrieved_docs, relevant_docs, k):
    """
    MAP@k (Mean Average Precision)를 계산합니다.

    Args:
        retrieved_docs: 검색된 문서 목록
        relevant_docs: 관련 문서 ID 집합
        k: 상위 몇 개 결과를 고려할지

    Returns:
        0~1 사이의 값 (각 정답 발견 시 precision의 평균)
    """
    # 상위 k개만 고려
    top_k_docs = retrieved_docs[:k] if k <= len(retrieved_docs) else retrieved_docs

    # 각 정답을 찾을 때마다의 Precision 계산
    precisions = []
    relevant_count = 0

    for i, doc in enumerate(top_k_docs):
        # 메타데이터에서 ID 추출 시도
        doc_id = doc['metadata'].get('id')

        # 메타데이터에 ID가 없으면 내용으로 찾기
        if not doc_id:
            doc_id = get_doc_id_by_content(doc['page_content'])

        # 정답에 포함되어 있는지 확인
        if doc_id in relevant_docs:
            relevant_count += 1
            precisions.append(relevant_count / (i + 1))

    # MAP 계산
    return sum(precisions) / len(relevant_docs) if precisions and relevant_docs else 0.0

In [ ]:
# 모든 메트릭을 계산하는 함수
def calculate_all_metrics(retrieved_docs, relevant_docs, k_values=[1, 3, 5, 10]):
    """
    모든 평가 메트릭을 계산하는 함수

    Args:
        retrieved_docs: 검색된 문서 목록
        relevant_docs: 관련 문서 ID 집합
        k_values: 평가할 k 값들의 리스트

    Returns:
        모든 메트릭 결과를 담은 딕셔너리
    """
    metrics = {}

    # 각 k 값에 대해 모든 메트릭 계산
    for k in k_values:
        metrics[f'Accuracy@{k}'] = calculate_accuracy_at_k(retrieved_docs, relevant_docs, k)
        metrics[f'Precision@{k}'] = calculate_precision_at_k(retrieved_docs, relevant_docs, k)
        metrics[f'Recall@{k}'] = calculate_recall_at_k(retrieved_docs, relevant_docs, k)
        metrics[f'MRR@{k}'] = calculate_mrr_at_k(retrieved_docs, relevant_docs, k)
        metrics[f'NDCG@{k}'] = calculate_ndcg_at_k(retrieved_docs, relevant_docs, k)
        metrics[f'MAP@{k}'] = calculate_map_at_k(retrieved_docs, relevant_docs, k)

    return metrics

In [ ]:
import pandas as pd

def evaluate_retriever(retriever, queries, relevant_docs, name="", k_values=[1, 3, 5, 10]):
    """
    각 리트리버의 성능을 평가하는 함수

    Args:
        retriever: 평가할 리트리버 객체
        queries: 질문 ID를 키로, 질문 텍스트를 값으로 하는 딕셔너리
        relevant_docs: 질문 ID를 키로, 관련 문서 ID 집합을 값으로 하는 딕셔너리
        name: 리트리버 이름 (출력용)
        k_values: 평가할 k 값들의 리스트

    Returns:
        평균 메트릭 딕셔너리
    """
    print(f"\n{name} 리트리버 평가 중...")
    results = []

    # 일부 질문만 평가 (속도 향상을 위해)
    # 실제 평가에서는 모든 질문을 사용하는 것이 더 정확합니다
    sample_queries = dict(list(queries.items())[:500])

    # 각 질문에 대해 리트리버 평가
    for query_id, query_text in tqdm(sample_queries.items()):
        # 리트리버로 문서 검색
        retrieved_docs = retriever.invoke(query_text)

        # 관련 문서 ID 가져오기
        expected_ids = relevant_docs.get(query_id, set())

        # 검색 결과 평가 (모든 메트릭 계산)
        metrics = calculate_all_metrics(retrieved_docs, expected_ids, k_values)

        # 결과 저장
        result = {
            'query_id': query_id,
            'query': query_text,
            **metrics
        }
        results.append(result)

    # 데이터프레임으로 변환
    df_results = pd.DataFrame(results)

    # 평균 메트릭 계산
    metrics_columns = [col for col in df_results.columns if any(col.startswith(prefix) for prefix in
                                                              ['Accuracy', 'Precision', 'Recall', 'MRR', 'NDCG', 'MAP'])]
    avg_metrics = df_results[metrics_columns].mean().to_dict()

    # 주요 메트릭 출력
    print(f"{name} 리트리버 평가 결과 (평균):")
    for k in sorted(k_values):
        print(f"  k={k} 결과:")
        for metric_prefix in ['Accuracy', 'Precision', 'Recall', 'MRR', 'NDCG', 'MAP']:
            metric_key = f"{metric_prefix}@{k}"
            if metric_key in avg_metrics:
                print(f"    {metric_key}: {avg_metrics[metric_key]:.4f}")

    return avg_metrics

In [ ]:
# 각 리트리버 평가
print("\n리트리버 평가 시작...")
embedding_metrics = evaluate_retriever(dense, queries, relevant_docs, "임베딩")
bm25_metrics = evaluate_retriever(bm25, queries, relevant_docs, "BM25")
bgem3_metrics = evaluate_retriever(bgem3, queries, relevant_docs, "bge-m3")
ensemble_metrics_1 = evaluate_retriever(ensemble1, queries, relevant_docs, "앙상블(bm25+dense)")
ensemble_metrics_2 = evaluate_retriever(ensemble2, queries, relevant_docs, "앙상블(bm25+bgem3)")


리트리버 평가 시작...

임베딩 리트리버 평가 중...


100%|██████████| 500/500 [03:33<00:00,  2.34it/s]


임베딩 리트리버 평가 결과 (평균):
  k=1 결과:
    Accuracy@1: 0.3500
    Precision@1: 0.3500
    Recall@1: 0.3500
    MRR@1: 0.3500
    NDCG@1: 0.3500
    MAP@1: 0.3500
  k=3 결과:
    Accuracy@3: 0.6080
    Precision@3: 0.2027
    Recall@3: 0.6080
    MRR@3: 0.4660
    NDCG@3: 0.5026
    MAP@3: 0.4660
  k=5 결과:
    Accuracy@5: 0.7240
    Precision@5: 0.1448
    Recall@5: 0.7240
    MRR@5: 0.4920
    NDCG@5: 0.5499
    MAP@5: 0.4920
  k=10 결과:
    Accuracy@10: 0.7240
    Precision@10: 0.1448
    Recall@10: 0.7240
    MRR@10: 0.4920
    NDCG@10: 0.5499
    MAP@10: 0.4920

BM25 리트리버 평가 중...


100%|██████████| 500/500 [00:00<00:00, 1015.86it/s]


BM25 리트리버 평가 결과 (평균):
  k=1 결과:
    Accuracy@1: 0.6040
    Precision@1: 0.6040
    Recall@1: 0.6040
    MRR@1: 0.6040
    NDCG@1: 0.6040
    MAP@1: 0.6040
  k=3 결과:
    Accuracy@3: 0.8140
    Precision@3: 0.2713
    Recall@3: 0.8140
    MRR@3: 0.7010
    NDCG@3: 0.7302
    MAP@3: 0.7010
  k=5 결과:
    Accuracy@5: 0.8620
    Precision@5: 0.1724
    Recall@5: 0.8620
    MRR@5: 0.7117
    NDCG@5: 0.7497
    MAP@5: 0.7117
  k=10 결과:
    Accuracy@10: 0.8620
    Precision@10: 0.1724
    Recall@10: 0.8620
    MRR@10: 0.7117
    NDCG@10: 0.7497
    MAP@10: 0.7117

bge-m3 리트리버 평가 중...


...

bge-m3 리트리버 평가 결과 (평균):
  k=1 결과:
    Accuracy@1: 0.6000
    Precision@1: 0.6000
    Recall@1: 0.6000
    MRR@1: 0.6000
    NDCG@1: 0.6000
    MAP@1: 0.6000
  k=3 결과:
    Accuracy@3: 0.7840
    Precision@3: 0.2613
    Recall@3: 0.7840
    MRR@3: 0.6843
    NDCG@3: 0.7101
    MAP@3: 0.6843
  k=5 결과:
    Accuracy@5: 0.8660
    Precision@5: 0.1732
    Recall@5: 0.8660
    MRR@5: 0.7027
    NDCG@5: 0.7435
    MAP@5: 0.7027
  k=10 결과:
    Accuracy@10: 0.8660
    Precision@10: 0.1732
    Recall@10: 0.8660
    MRR@10: 0.7027
    NDCG@10: 0.7435
    MAP@10: 0.7027

앙상블(bm25+dense) 리트리버 평가 중...


100%|██████████| 500/500 [03:27<00:00,  2.41it/s]


앙상블(bm25+dense) 리트리버 평가 결과 (평균):
  k=1 결과:
    Accuracy@1: 0.5560
    Precision@1: 0.5560
    Recall@1: 0.5560
    MRR@1: 0.5560
    NDCG@1: 0.5560
    MAP@1: 0.5560
  k=3 결과:
    Accuracy@3: 0.8440
    Precision@3: 0.2813
    Recall@3: 0.8440
    MRR@3: 0.6837
    NDCG@3: 0.7249
    MAP@3: 0.6837
  k=5 결과:
    Accuracy@5: 0.9120
    Precision@5: 0.1824
    Recall@5: 0.9120
    MRR@5: 0.7000
    NDCG@5: 0.7535
    MAP@5: 0.7000
  k=10 결과:
    Accuracy@10: 0.9120
    Precision@10: 0.1824
    Recall@10: 0.9120
    MRR@10: 0.7000
    NDCG@10: 0.7535
    MAP@10: 0.7000

앙상블(bm25+bgem3) 리트리버 평가 중...


100%|██████████| 500/500 [04:50<00:00,  1.72it/s]

앙상블(bm25+bgem3) 리트리버 평가 결과 (평균):
  k=1 결과:
    Accuracy@1: 0.6580
    Precision@1: 0.6580
    Recall@1: 0.6580
    MRR@1: 0.6580
    NDCG@1: 0.6580
    MAP@1: 0.6580
  k=3 결과:
    Accuracy@3: 0.8880
    Precision@3: 0.2960
    Recall@3: 0.8880
    MRR@3: 0.7633
    NDCG@3: 0.7955
    MAP@3: 0.7633
  k=5 결과:
    Accuracy@5: 0.9180
    Precision@5: 0.1836
    Recall@5: 0.9180
    MRR@5: 0.7704
    NDCG@5: 0.8081
    MAP@5: 0.7704
  k=10 결과:
    Accuracy@10: 0.9180
    Precision@10: 0.1836
    Recall@10: 0.9180
    MRR@10: 0.7704
    NDCG@10: 0.8081
    MAP@10: 0.7704


In [ ]:
results = {
    '임베딩': embedding_metrics,
    'BM25': bm25_metrics,
    'bge-m3': bgem3_metrics,
    '앙상블(bm25+dense)': ensemble_metrics_1,
    '앙상블(bm25+bgem3)': ensemble_metrics_2
}

# 데이터프레임으로 변환 (전치하여 메트릭을 행으로 변환)
results_df = pd.DataFrame(results)

print("\n리트리버 성능 비교:")
print(results_df)

# 결과 저장
results_df.to_csv("retriever_comparison_results.csv")
print("평가 결과를 'retriever_comparison_results.csv'에 저장했습니다.")


리트리버 성능 비교:
                   임베딩      BM25    bge-m3  앙상블(bm25+dense)  앙상블(bm25+bgem3)
Accuracy@1    0.350000  0.604000  0.600000         0.556000         0.658000
Precision@1   0.350000  0.604000  0.600000         0.556000         0.658000
Recall@1      0.350000  0.604000  0.600000         0.556000         0.658000
MRR@1         0.350000  0.604000  0.600000         0.556000         0.658000
NDCG@1        0.350000  0.604000  0.600000         0.556000         0.658000
MAP@1         0.350000  0.604000  0.600000         0.556000         0.658000
Accuracy@3    0.608000  0.814000  0.784000         0.844000         0.888000
Precision@3   0.202667  0.271333  0.261333         0.281333         0.296000
Recall@3      0.608000  0.814000  0.784000         0.844000         0.888000
MRR@3         0.466000  0.701000  0.684333         0.683667         0.763333
NDCG@3        0.502567  0.730211  0.710068         0.724877         0.795520
MAP@3         0.466000  0.701000  0.684333         0.683667    

## 실제 출력 비교

In [ ]:
def analyze_all_retrievers(queries, corpus, relevant_docs, bm25_retriever, embedding_retriever, bgem3_retriever, ensemble_retriever_1, ensemble_retriever_2, num_samples=5, top_k=3):
    """
    5가지 리트리버(BM25, 임베딩, bgem3, 앙상블1(bm25+dense), 앙상블2(bm25+bgem3))의 검색 결과를 비교 분석하는 함수

    Args:
        queries: 질문 ID를 키로, 질문 텍스트를 값으로 하는 딕셔너리
        corpus: 문서 ID를 키로, 문서 텍스트를 값으로 하는 딕셔너리
        relevant_docs: 질문 ID를 키로, 관련 문서 ID 집합을 값으로 하는 딕셔너리
        bm25_retriever: BM25 리트리버 객체
        embedding_retriever: 임베딩 리트리버 객체
        bgem3_retriever : bgem3 리트리버 객체
        ensemble_retriever_1: 앙상블1 리트리버 객체
        ensemble_retriever_2: 앙상블2 리트리버 객체
        num_samples: 분석할 질문 수
        top_k: 출력할 상위 검색 결과 수
    """
    print(f"\n===== 리트리버 검색 결과 비교 분석 (샘플 {num_samples}개) =====\n")

    # 샘플 질문 선택
    sample_query_ids = list(queries.keys())[:num_samples]

    # 내용으로 문서 ID 찾는 함수
    def find_doc_id_by_content(content):
        for doc_id, doc_content in corpus.items():
            if content == doc_content or content in doc_content or doc_content in content:
                return doc_id
        return None

    # 각 질문에 대해 분석
    for idx, query_id in enumerate(sample_query_ids):
        query_text = queries[query_id]                        # 질문 내용
        expected_doc_ids = relevant_docs.get(query_id, set()) # 참조 doc ID

        print(f"\n{'='*80}")
        print(f"\n[질문 {idx+1}] {query_text}")
        print('--' * 20)

        # 정답 문서 내용 출력
        print("\n💡 정답 문서:")
        for doc_id in expected_doc_ids:
            doc_text = corpus.get(doc_id, "문서를 찾을 수 없음")
            print(f"  문서 ID: {doc_id}")
            print(f"  내용: {doc_text[:150]}..." if len(doc_text) > 150 else doc_text)

        # 각 리트리버별 검색 결과 출력
        retrievers = {
            "BM25": bm25_retriever,
            "임베딩": embedding_retriever,
            "bge-m3": bgem3_retriever,
            "앙상블(bm25+dense)": ensemble_retriever_1,
            "앙상블(bm25+bgem3)": ensemble_retriever_2
        }
        print('==' * 50)
        for name, retriever in retrievers.items():
            # 검색 수행
            retrieved_docs = retriever.invoke(query_text)

            # 검색 결과 출력
            print(f"\n🔍 {name} 리트리버 검색 결과 (상위 {top_k}개):")
            for i, doc in enumerate(retrieved_docs[:top_k]):
                # 문서 ID 확인 (메타데이터 또는 내용 기반)
                doc_id = doc['metadata'].get('id', None)
                if not doc_id or doc_id.startswith("unknown"):
                    doc_id = find_doc_id_by_content(doc['page_content']) or f"unknown_{i}"

                # 검색된 문서가 질문 생성 시 인용된 문서들에 해당되는지
                is_relevant = doc_id in expected_doc_ids
                relevance_mark = "✓" if is_relevant else "✗"

                print(f"  [{i+1}] {relevance_mark} 문서 ID: {doc_id}")
                print(f"      내용: {doc['page_content'][:150]}..." if len(doc['page_content']) > 150 else doc['page_content'])
                print('--' * 20)

            # 정확도 계산
            correct_in_top_k = 0
            for doc in retrieved_docs[:top_k]:
                doc_id = doc['metadata'].get('id', None)
                if not doc_id or doc_id.startswith("unknown"):
                    doc_id = find_doc_id_by_content(doc['page_content'])

                if doc_id in expected_doc_ids:
                    correct_in_top_k += 1

            accuracy = correct_in_top_k / min(top_k, len(retrieved_docs)) if retrieved_docs else 0
            print(f"  정확도: {correct_in_top_k}/{min(top_k, len(retrieved_docs))} ({accuracy:.2f})")

        # 키워드 분석
        print("\n🔑 질문-문서 키워드 분석:")
        query_words = set(query_text.lower().split())

        # 정답 문서의 키워드
        answer_doc_id = next(iter(expected_doc_ids)) if expected_doc_ids else None
        if answer_doc_id:
            answer_doc_text = corpus.get(answer_doc_id, "")
            answer_words = set(answer_doc_text.lower().split())
            common_words = query_words.intersection(answer_words)

            print(f"  질문 키워드: {', '.join(query_words)}")
            print(f"  정답 문서와 공통 키워드: {', '.join(common_words)}")

        print(f"\n{'='*80}")

# 함수 실행
analyze_all_retrievers(
    queries,
    corpus,
    relevant_docs,
    bm25,
    dense,
    bgem3,
    ensemble1,
    ensemble2
)


===== 리트리버 검색 결과 비교 분석 (샘플 5개) =====



[질문 1] 대한민국 인공지능 행동계획에서 제시된 첫 번째 정책축의 주요 목표는 무엇인가요?
----------------------------------------

💡 정답 문서:
  문서 ID: doc_0
  내용: 대한민국 인공지능 행동계획
2026. 2. 25
국가인공지능전략위원회
- 2 -
≪들어가며≫인공지능(AI)은 과학적 발견의 방식을 근본적으로 바꾸고, 산업의 구조를 다시 짜며, 인간의 사고와 소통, 학습의 영역까지 재정의하고 있습니다. 이 거대한 기술혁명은 세계 질서를...

🔍 BM25 리트리버 검색 결과 (상위 3개):
  [1] ✓ 문서 ID: doc_0
      내용: 대한민국 인공지능 행동계획
2026. 2. 25
국가인공지능전략위원회
- 2 -
≪들어가며≫인공지능(AI)은 과학적 발견의 방식을 근본적으로 바꾸고, 산업의 구조를 다시 짜며, 인간의 사고와 소통, 학습의 영역까지 재정의하고 있습니다. 이 거대한 기술혁명은 세계 질서를...
----------------------------------------
  [2] ✗ 문서 ID: doc_82
      내용: 법으로 따로 정하지 않는 한 공개한다. 공무원들은 언제든 일을 하는 데 필요한 최고성능의, 가장 적합한 AI를 제공받는다.   국민은 AI를 활용한 최고의 대국민 서비스를 받는다. AI를 활용한 최고의 대국민 서비스는 ; 정부의 모든 서비스는 하나의 아이디로 쓸 수 있...
----------------------------------------
  [3] ✗ 문서 ID: doc_1
      내용:  핵심인재 확보, ④ 독자 범용 인공지능 모델 확보, ⑤ 인공지능 규제혁신입니다. 이 5가지 분야는 인공지능 강국의 토대를 마련하고, 다음 세대의 국가 경쟁력을 결정짓는 출발점이 될 것입니다.
- 3 -
두 번째 축은 범국가 인공지능 기반 대전환입니다.인공지능은 더 이...
---------------